# VQA Fine-tuning Improved

RTX 5060 Ti + Qwen2.5-VL-3B-Instruct 기준 개선본입니다.

주요 변경점:
- 전체 학습 데이터 사용(기본값)
- train/validation 분리 시 정답 비율을 최대한 유지
- **assistant 정답 부분에만 loss 적용**
- 이미지 해상도 상향(텍스트 이미지 인식 보강)
- LoRA 용량 상향(r=16)
- learning rate 2e-4 + cosine scheduler + warmup
- 3 epoch 학습
- 매 epoch validation accuracy 측정
- validation accuracy가 가장 좋은 LoRA 파라미터 복원
- 추론 시 새로 생성된 토큰만 decode
- sample_submission.csv 형식을 그대로 사용해 submission.csv 생성

> 처음 실행할 때는 VS Code에서 **Kernel Restart 후 Run All**을 권장합니다.

# 1. 환경 준비

In [ ]:
print('hello123')

In [ ]:
%pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130

In [ ]:
%pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas tqdm --upgrade

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

# 2. 라이브러리 / 데이터 / 설정

In [ ]:
import os, re, math, random
from pathlib import Path
from dataclasses import dataclass
from typing import Any

import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# 텍스트가 포함된 이미지라 baseline 384보다 조금 높임.
# CUDA OOM이 나면 512 -> 448 또는 384로 낮추세요.
IMAGE_SIZE = 512
MAX_NEW_TOKENS = 4

VALID_RATIO = 0.10
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

# None이면 전체 데이터 사용. 빠른 동작 확인만 할 때 200 같은 숫자로 바꾸면 됩니다.
DEBUG_SAMPLE_N = None

OUTPUT_ROOT = Path("outputs")
BEST_MODEL_DIR = OUTPUT_ROOT / "qwen2_5_vl_3b_lora_best"
SUBMISSION_PATH = Path("submission.csv")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
print("device:", device)
print("compute dtype:", COMPUTE_DTYPE)

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

required_train_cols = {"path", "question", "a", "b", "c", "d", "answer"}
required_test_cols = {"path", "question", "a", "b", "c", "d", "id"}
assert required_train_cols.issubset(train_df.columns), f"train.csv 컬럼 확인 필요: {train_df.columns.tolist()}"
assert required_test_cols.issubset(test_df.columns), f"test.csv 컬럼 확인 필요: {test_df.columns.tolist()}"

train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df["answer"].unique()).issubset({"a","b","c","d"}), "answer에는 a/b/c/d만 있어야 합니다."

if DEBUG_SAMPLE_N is not None:
    n = min(DEBUG_SAMPLE_N, len(train_df))
    train_df = train_df.sample(n=n, random_state=SEED).reset_index(drop=True)

print("train:", len(train_df), "test:", len(test_df))
print(train_df["answer"].value_counts(normalize=True).sort_index())

# 3. Train / Validation 분리

In [ ]:
def stratified_split(df, valid_ratio=0.1, seed=42):
    train_parts = []
    valid_parts = []

    for _, group in df.groupby("answer"):
        group = group.sample(frac=1, random_state=seed)
        if len(group) <= 1:
            train_parts.append(group)
            continue

        n_valid = max(1, int(round(len(group) * valid_ratio)))
        n_valid = min(n_valid, len(group) - 1)
        valid_parts.append(group.iloc[:n_valid])
        train_parts.append(group.iloc[n_valid:])

    train_part = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    valid_part = pd.concat(valid_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train_part, valid_part

train_subset, valid_subset = stratified_split(train_df, VALID_RATIO, SEED)

print("train subset:", len(train_subset))
print("valid subset:", len(valid_subset))
print("valid answer ratio:")
print(valid_subset["answer"].value_counts(normalize=True).sort_index())

# 4. 모델 / Processor / QLoRA

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# 학습 시 label masking을 단순하고 안전하게 만들기 위해 오른쪽 padding 사용
processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# quantized model + device_map='auto'에서는 model.to(device)를 다시 호출하지 않습니다.
MODEL_DEVICE = next(model.parameters()).device
print("model input device:", MODEL_DEVICE)

# 5. 프롬프트

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a visual question answering assistant. "
    "Inspect the image carefully, especially all visible text. "
    "Choose the single correct option. "
    "Return exactly one lowercase letter: a, b, c, or d. "
    "Do not explain."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Answer with exactly one lowercase letter among a, b, c, d."
    )

# 6. Dataset / Collator

핵심 개선점입니다. 기존 baseline은 입력 프롬프트 전체를 `labels`로 복사해서 **질문과 선택지까지 전부 예측하도록 loss를 계산**했습니다.

아래 collator는 **assistant의 정답 부분에만 loss가 걸리도록** 나머지 토큰을 `-100`으로 마스킹합니다.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_mc_prompt(
            str(row["question"]), str(row["a"]), str(row["b"]),
            str(row["c"]), str(row["d"])
        )

        base_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ]},
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            full_messages = base_messages + [
                {"role": "assistant", "content": [{"type": "text", "text": gold}]}
            ]
            return {
                "full_messages": full_messages,
                "prompt_messages": base_messages,
                "image": img,
                "answer": gold,
            }

        return {
            "prompt_messages": base_messages,
            "image": img,
        }


@dataclass
class TrainDataCollator:
    processor: Any

    def __call__(self, batch):
        full_texts = []
        prompt_texts = []
        images = []

        for sample in batch:
            full_text = self.processor.apply_chat_template(
                sample["full_messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            prompt_text = self.processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True,
            )
            full_texts.append(full_text)
            prompt_texts.append(prompt_text)
            images.append(sample["image"])

        enc = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        # 같은 이미지 + prompt-only를 토큰화해서 assistant 답변 시작 위치를 구함
        prompt_enc = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        prompt_lengths = prompt_enc["attention_mask"].sum(dim=1).tolist()

        labels = enc["input_ids"].clone()

        # padding은 loss에서 제외
        labels[enc["attention_mask"] == 0] = -100

        # system + image + question + options + assistant header는 loss에서 제외
        for i, prompt_len in enumerate(prompt_lengths):
            labels[i, :int(prompt_len)] = -100

        enc["labels"] = labels
        return enc

# 7. DataLoader

In [ ]:
train_ds = VQAMCDataset(train_subset, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=TrainDataCollator(processor),
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("train batches:", len(train_loader))

# 8. 추론 함수 / Validation Accuracy

In [ ]:
CHOICES = {"a", "b", "c", "d"}

def extract_choice(text: str) -> str:
    text = text.strip().lower()

    if text in CHOICES:
        return text

    # 단독 문자 a/b/c/d 우선 탐색
    m = re.search(r"(?<![a-z])([abcd])(?![a-z])", text)
    if m:
        return m.group(1)

    # 제출 형식 보장을 위한 fallback
    return "a"


def predict_row(row):
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ]},
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text], images=[img], return_tensors="pt"
    )
    inputs = {k: v.to(MODEL_DEVICE) for k, v in inputs.items()}

    with torch.inference_mode():
        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available(),
            dtype=COMPUTE_DTYPE,
        ):
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

    # 입력 프롬프트는 버리고 새로 생성된 토큰만 decode
    generated_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    generated_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]

    return extract_choice(generated_text), generated_text


def evaluate_accuracy(df):
    model.eval()
    correct = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Validation", unit="sample"):
        pred, _ = predict_row(row)
        gold = str(row["answer"]).strip().lower()
        correct += int(pred == gold)

    return correct / max(1, len(df))

# 9. Fine-tuning

- learning rate: `2e-4`
- epoch: `3`
- gradient accumulation: `4`
- 매 epoch validation accuracy 측정
- 가장 높은 validation accuracy의 **LoRA 파라미터만 CPU에 보관**했다가 학습 종료 후 복원

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * updates_per_epoch
num_warmup_steps = max(1, int(num_training_steps * WARMUP_RATIO))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

use_scaler = torch.cuda.is_available() and COMPUTE_DTYPE == torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)

best_val_acc = -1.0
best_epoch = -1
best_trainable_state = None

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    seen_batches = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} [train]",
        unit="batch",
    )

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(MODEL_DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available(),
            dtype=COMPUTE_DTYPE,
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss
            loss = raw_loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        running_loss += raw_loss.item()
        seen_batches += 1

        should_step = (step % GRAD_ACCUM == 0) or (step == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        progress_bar.set_postfix({
            "loss": f"{running_loss / seen_batches:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
        })

    # 대회 평가 기준이 Accuracy이므로 epoch마다 validation accuracy 확인
    model.config.use_cache = True
    val_acc = evaluate_accuracy(valid_subset)
    model.config.use_cache = False

    avg_train_loss = running_loss / max(1, seen_batches)
    print(
        f"[Epoch {epoch+1}] "
        f"train loss={avg_train_loss:.4f} | valid accuracy={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_trainable_state = {
            name: param.detach().cpu().clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }
        print(f"  -> best updated: epoch {best_epoch}, acc={best_val_acc:.4f}")

# best LoRA 파라미터 복원
if best_trainable_state is not None:
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_trainable_state:
                param.copy_(best_trainable_state[name].to(param.device, dtype=param.dtype))

print(f"Best epoch: {best_epoch}, best validation accuracy: {best_val_acc:.4f}")

model.config.use_cache = True
model.eval()

BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(BEST_MODEL_DIR))
processor.save_pretrained(str(BEST_MODEL_DIR))
print("Saved best adapter:", BEST_MODEL_DIR.resolve())

# 10. Test inference + submission.csv 생성

In [ ]:
model.eval()
preds = []
raw_outputs = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference", unit="sample"):
    pred, raw_text = predict_row(row)
    preds.append(pred)
    raw_outputs.append(raw_text)

# sample_submission.csv가 있으면 그 형식을 그대로 사용
if Path("sample_submission.csv").exists():
    submission = pd.read_csv("sample_submission.csv")
    assert len(submission) == len(preds), "sample_submission.csv와 test.csv 행 수가 다릅니다."
    submission["answer"] = preds
else:
    submission = pd.DataFrame({
        "id": test_df["id"].values,
        "answer": preds,
    })

submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH.resolve())
print("rows:", len(submission))
print(submission.head())
print("\npred distribution:")
print(submission["answer"].value_counts().sort_index())

# 11. 추론 결과 간단 확인

In [ ]:
check_n = min(10, len(test_df))
preview = pd.DataFrame({
    "id": test_df["id"].iloc[:check_n].values,
    "pred": preds[:check_n],
    "raw_output": raw_outputs[:check_n],
})
preview

## 실행 결과

모든 셀이 오류 없이 끝까지 실행되면 현재 작업 폴더에 **`submission.csv`**가 생성됩니다.

또한 가장 validation accuracy가 좋았던 LoRA 어댑터는 `outputs/qwen2_5_vl_3b_lora_best/`에 저장됩니다.

메모리 부족(`CUDA out of memory`)이 발생하면 가장 먼저 `IMAGE_SIZE = 512`를 `448` 또는 `384`로 낮추세요.